# Handling Missing Values & Data Cleaning

This notebook covers the **missing-value treatment and data-cleaning** step that follows the exploratory data analysis performed in the previous notebook. The goal is to produce a clean dataset that can be fed directly into feature engineering and model training.

**Why this matters:** Most machine-learning algorithms (e.g., scikit-learn estimators) cannot handle `NaN` values natively. Leaving missing values untreated will cause errors at training time or, worse, silently produce wrong results. Similarly, invalid entries distort summary statistics and can mislead the model. Addressing these issues now ensures a smooth downstream pipeline.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


## Loading the Data & Assessing Missingness

We start by loading the original `spitogatos.csv` file and computing the **proportion of missing values per column**. This recap of the EDA quality check reminds us which features need attention and helps us prioritise: features with a very high missing rate may need a different strategy (or even removal) compared to features with only a few gaps.


In [2]:
df = pd.read_csv("spitogatos.csv")
df.head()


,listingId,airConditioning,alarm,areaSqm,balcony,bathrooms,closedParkingSpots,elevator,energyClass,fireplace,...,pool,price,pricePerSqm,propertyType,renovationYear,rooms,solarHeater,storage,view,yearBuilt
0,20304396,True,False,47,False,1.0,NaN,True,C,False,...,False,600,12.77,apartment,2026.0,NaN,False,False,True,1975.0
1,19956518,True,False,25,False,1.0,NaN,True,B,False,...,False,530,21.20,apartment,2023.0,NaN,False,False,True,1982.0
2,18383741,False,False,60,False,1.0,NaN,True,D,False,...,False,510,8.50,apartment,NaN,1.0,False,False,True,1978.0
3,20242029,True,False,60,False,1.0,NaN,True,F,False,...,False,330,5.50,apartment,NaN,1.0,False,False,True,1978.0
4,20055845,True,False,75,False,1.0,NaN,True,C,False,...,False,620,8.27,apartment,NaN,2.0,False,False,True,1981.0


In [3]:
print(round(df.isnull().sum()/len(df),3))


listingId             0.000
airConditioning       0.035
alarm                 0.071
areaSqm               0.000
balcony               0.031
bathrooms             0.026
closedParkingSpots    0.998
elevator              0.025
energyClass           0.001
fireplace             0.067
floor                 0.000
furnished             0.031
garden                0.076
hasParking            0.061
isNewDevelopment      0.000
kitchens              0.075
listingType           0.000
livingRooms           0.169
location              0.000
openParkingSpots      0.967
petsAllowed           0.894
pool                  0.076
price                 0.000
pricePerSqm           0.000
propertyType          0.000
renovationYear        0.530
rooms                 0.096
solarHeater           0.073
storage               0.082
view                  0.061
yearBuilt             0.098
dtype: float64


## Filling Missing Values by Feature Type

We are going to fill missing values specifically for each feature. The imputation strategy depends on the nature of the feature:

- **Binary/boolean features** (`airConditioning`, `alarm`, `balcony`, `elevator`, `fireplace`, `garden`, `hasParking`, `petsAllowed`, `pool`, `solarHeater`, `storage`, `view`): filled with `'False'`. The reasoning is that if a listing does not mention an amenity, it is reasonable to assume the apartment does not have it. This is a conservative assumption that avoids inflating the perceived value of a property.
- **Count features** (`bathrooms`, `kitchens`, `livingRooms`, `rooms`): filled with `1`, the most common and minimum sensible value for an apartment.
- **Parking spots** (`closedParkingSpots`, `openParkingSpots`): filled with `0`, meaning no parking.
- **Furnish status** (`furnished`): filled with `'no'` (unfurnished), the most conservative assumption.
- **Energy class** (`energyClass`): filled with `'C'`, which was identified in EDA as the most frequent class, representing moderate energy efficiency.


In [4]:
# Fill missing values based on the feature type
df['airConditioning'] = df['airConditioning'].fillna('False')
df['alarm'] = df['alarm'].fillna('False')
df['balcony'] = df['balcony'].fillna('False')
df['bathrooms'] = df['bathrooms'].fillna(1)
df['closedParkingSpots'] = df['closedParkingSpots'].fillna(0)
df['elevator'] = df['elevator'].fillna('False')
df['energyClass'] = df['energyClass'].fillna('C')
df['fireplace'] = df['fireplace'].fillna('False')
df['furnished'] = df['furnished'].fillna('no')
df['garden'] = df['garden'].fillna('False')
df['hasParking'] = df['hasParking'].fillna('False')
df['kitchens'] = df['kitchens'].fillna(1)
df['livingRooms'] = df['livingRooms'].fillna(1)
df['openParkingSpots'] = df['openParkingSpots'].fillna(0)
df['petsAllowed'] = df['petsAllowed'].fillna('False')
df['pool'] = df['pool'].fillna('False')
df['rooms'] = df['rooms'].fillna(1)
df['solarHeater'] = df['solarHeater'].fillna('False')
df['storage'] = df['storage'].fillna('False')
df['view'] = df['view'].fillna('False')


## Special Case: `renovationYear` (53% Missing)

The `renovationYear` column is different from the features above: roughly 53% of its values are missing. Filling over half the data with a single value (mean, median, or mode) would introduce significant noise and could mask a genuine signal.

A better approach is to treat the *absence* of a renovation year as potentially informative in itself. We create a **binary indicator variable** (`renovationYear_ind`): `1` if the renovation year is known, `0` if it is missing. By comparing the average rent for each group using a pivot table, we can determine whether the missingness carries any predictive information.


In [5]:
df['renovationYear'].value_counts()


renovationYear
2026.0    260
2025.0    177
2024.0    131
2020.0    104
2023.0     99
2022.0     87
2021.0     42
2019.0     41
2018.0     40
2010.0     40
2015.0     32
2016.0     24
2000.0     21
2017.0     10
2014.0     10
2012.0      9
2005.0      9
2006.0      4
2008.0      4
2009.0      4
2007.0      3
1990.0      3
2013.0      2
2002.0      2
2011.0      2
1981.0      2
2001.0      1
1972.0      1
1998.0      1
2003.0      1
Name: count, dtype: int64

In [6]:
df_copy = df.copy()
df_copy['renovationYear_ind'] = np.where(df_copy['renovationYear'].isnull(), 0, 1)
df_copy.pivot_table('price', index='renovationYear_ind', aggfunc="mean")


,price
renovationYear_ind,
0,703.926941
1,600.793310


We notice that the mean price for apartments where the renovation year is not registered tends to be higher. This is an interesting finding: it suggests that newer or recently renovated apartments (whose renovation year is explicitly listed) may actually rent for *less* on average, or conversely that the *absence* of a renovation year correlates with older, more expensive properties. Later on in the feature engineering phase we can create the binary feature described above to capture this signal. For now, we leave the original `renovationYear` column as is.


## Column `yearBuilt` (Invalid Values)

During EDA we identified two types of problems with `yearBuilt`: **missing values** and **invalid entries**. We handle them separately.

First, we inspect the distribution of existing values. Then we impute the missing values with the **median** year built. The median is preferred over the mean here because it is robust to outliers — if a few extreme values are present, the median remains unaffected whereas the mean would be pulled toward those extremes.


In [7]:
df['yearBuilt'].value_counts()


yearBuilt
1970.0    383
1980.0    210
1975.0    175
1965.0    105
1978.0     80
         ... 
1951.0      1
1938.0      1
1922.0      1
1952.0      1
1933.0      1
Name: count, Length: 84, dtype: int64

In [8]:
df['yearBuilt'] = df['yearBuilt'].fillna(df['yearBuilt'].median())


Earlier in EDA we observed that the `yearBuilt` variable contains some entries equal to **1000**. This is clearly a data-entry error. A plausible explanation is that the listing author intended to type `2000` and missed the leading `2`. We therefore replace all instances of `1000` with `2000`.


In [9]:
df[df['yearBuilt'] == 1000].head()


,listingId,airConditioning,alarm,areaSqm,balcony,bathrooms,closedParkingSpots,elevator,energyClass,fireplace,...,pool,price,pricePerSqm,propertyType,renovationYear,rooms,solarHeater,storage,view,yearBuilt
2158,20187385,True,False,90,True,1.0,0.0,True,D,False,...,False,850,9.44,apartment,NaN,3.0,False,False,False,1000.0
2176,20212856,True,False,40,False,1.0,0.0,True,F,False,...,False,420,10.50,apartment,2025.0,1.0,False,False,False,1000.0


In [10]:
df['yearBuilt'] = df['yearBuilt'].replace(1000, 2000)


After the replacement, we verify that no rows with `yearBuilt == 1000` remain.


In [11]:
df[df['yearBuilt'] == 1000].head()


,listingId,airConditioning,alarm,areaSqm,balcony,bathrooms,closedParkingSpots,elevator,energyClass,fireplace,...,pool,price,pricePerSqm,propertyType,renovationYear,rooms,solarHeater,storage,view,yearBuilt


Next, we check for entries where `yearBuilt` equals **2155**. This is another obvious error because it is far in the future and cannot represent a real construction year. Unlike the `1000` case, we cannot confidently guess the intended value (it could be 2015, 2005, 1955, etc.), so the safest approach is to **drop these rows entirely** rather than introduce a potentially incorrect value.


In [12]:
df[df['yearBuilt'] == 2155].head()


,listingId,airConditioning,alarm,areaSqm,balcony,bathrooms,closedParkingSpots,elevator,energyClass,fireplace,...,pool,price,pricePerSqm,propertyType,renovationYear,rooms,solarHeater,storage,view,yearBuilt
613,19945100,True,False,48,False,1.0,0.0,False,C,False,...,False,650,13.54,apartment,NaN,1.0,False,False,False,2155.0
1108,20295487,True,False,60,True,1.0,0.0,False,C,False,...,False,600,10.00,apartment,NaN,2.0,False,False,False,2155.0
1217,20211719,False,False,57,True,1.0,0.0,True,A+,False,...,False,750,13.16,apartment,NaN,1.0,False,True,True,2155.0
1398,19925360,False,False,74,True,1.0,0.0,True,C,False,...,False,650,8.78,apartment,NaN,2.0,False,False,False,2155.0
1402,19891443,True,False,58,True,1.0,0.0,False,B,False,...,False,600,10.34,apartment,NaN,1.0,False,False,False,2155.0


In [13]:
i = df[df['yearBuilt'] == 2155].index
df = df.drop(i)


We verify that no rows with `yearBuilt == 2155` remain in the dataset.


In [14]:
df[df['yearBuilt'] == 2155].head()


,listingId,airConditioning,alarm,areaSqm,balcony,bathrooms,closedParkingSpots,elevator,energyClass,fireplace,...,pool,price,pricePerSqm,propertyType,renovationYear,rooms,solarHeater,storage,view,yearBuilt


## Final Verification

Before saving the cleaned dataset, we perform a final check by calling `df.isnull().sum()`. At this point, the only column that should still contain missing values is `renovationYear`, which we intentionally left as is because we plan to handle it through a binary indicator feature during feature engineering. All other columns should show `0` missing values.


In [15]:
df.isnull().sum()


listingId                0
airConditioning          0
alarm                    0
areaSqm                  0
balcony                  0
bathrooms                0
closedParkingSpots       0
elevator                 0
energyClass              0
fireplace                0
floor                    0
furnished                0
garden                   0
hasParking               0
isNewDevelopment         0
kitchens                 0
listingType              0
livingRooms              0
location                 0
openParkingSpots         0
petsAllowed              0
pool                     0
price                    0
pricePerSqm              0
propertyType             0
renovationYear        1306
rooms                    0
solarHeater              0
storage                  0
view                     0
yearBuilt                0
dtype: int64

## Saving the Cleaned Dataset

We export the cleaned DataFrame to `spitogatos_no_missing.csv`. This file will be loaded by the subsequent notebooks for outlier removal, feature engineering, and model training. Keeping each stage's output in a separate file makes the pipeline reproducible and easy to debug. If something goes wrong downstream, we can always trace it back to the last clean checkpoint.


In [16]:
df.to_csv("spitogatos_no_missing.csv", index=False)
